# 05 - Indici dell'errore, commutazioni dell'infusione e sensibilità ai parametri di tetto IOB e PLGS

**Obiettivo.** Rispondere alle richieste della co-relatrice dopo la revisione del repository:
1. aggiungere un indice dell'errore (IAE) e il tempo di rientro nel range;
2. quantificare le commutazioni e la variabilità dell'infusione introdotte dai vincoli;
3. fare una breve analisi di sensibilità sui parametri principali del tetto IOB e del PLGS.

**Scenario.** Pasto non annunciato alle 12:00 da 60 e 90 g su `adult#001` e `adult#003` (i due pazienti in cui il
controllore proposto ha effetti opposti), stesso seme del sensore, stesse condizioni iniziali e assestamento del notebook 02.
I moduli sono attivati separatamente (`PID+IOB`, `PID+PLGS`) e insieme (`PID+IOB+PLGS`) a parità di PID.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import aid_sim as A, notebook_utils as U
import run_campaign as RC          # stesse funzioni, stessa cache e simulazioni in parallelo della campagna
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 110); pd.set_option("display.max_columns", 40)
SEED = 1                                  # seme del rumore del sensore (uguale per tutti i controllori)
COL = RC.COL
print("Versioni delle librerie usate:"); display(U.library_versions())

Versioni delle librerie usate:


,Pacchetto,Versione
0,Python,3.13.16
1,simglucose,0.2.11
2,numpy,2.4.4
3,scipy,1.17.1
4,pandas,3.0.2
5,matplotlib,3.10.8


## 1. Configurazione
### 1.1 Scenario: condizioni iniziali, durata, passo, pasto, seme

In [2]:
PAZ = RC.SENS_PAZ; CARICHI = RC.SENS_CHO
for p in PAZ:
    print(p); display(U.scenario_table(p, "60 o 90", SEED))

adult#001


,Voce,Valore,Unita'
0,Paziente virtuale,adult#001,-
1,Peso corporeo BW,102.3,kg
2,Glicemia basale di equilibrio Gb,138.6,mg/dL
3,Fabbisogno basale nominale u_b,1.267,U/h
4,Sensore CGM,Dexcom (rumore di simglucose),-
5,Pompa di infusione,Insulet,-
6,Seme casuale del sensore,1,-
7,Inizio simulazione,00:00,ora del giorno
8,Assestamento escluso dalle metriche,00:00-08:00,ora del giorno
9,Finestra di analisi,08:00-24:00,ora del giorno


adult#003


,Voce,Valore,Unita'
0,Paziente virtuale,adult#003,-
1,Peso corporeo BW,81.6,kg
2,Glicemia basale di equilibrio Gb,147.1,mg/dL
3,Fabbisogno basale nominale u_b,1.425,U/h
4,Sensore CGM,Dexcom (rumore di simglucose),-
5,Pompa di infusione,Insulet,-
6,Seme casuale del sensore,1,-
7,Inizio simulazione,00:00,ora del giorno
8,Assestamento escluso dalle metriche,00:00-08:00,ora del giorno
9,Finestra di analisi,08:00-24:00,ora del giorno


### 1.2 Parametri e vincoli del controllore proposto (con unità di misura)

In [3]:
ub = U.initial_conditions("adult#001", SEED)["ub"]
display(U.controller_table(A.CFG["PID+IOB+PLGS"], ub))

,Parametro,Valore,Unita',Significato
0,target,120.0,mg/dL,Glicemia obiettivo del regolatore
1,kp,0.0001,U/min per mg/dL,Guadagno proporzionale
2,ki,0.000005,U/min per (mg/dL*min),Guadagno integrale
3,kd,0.002,U/min per (mg/dL/min),Guadagno derivativo sulla pendenza CGM
4,n_slope,5,campioni,Campioni CGM usati per stimare la pendenza ai minimi quadrati
5,aw_frac,1.0,adimensionale,Anti-windup: Ki*I_max = aw_frac * basale
6,aw_frac_low,0.25,adimensionale,Anti-windup: Ki*I_min = -aw_frac_low * basale
7,freeze,True,-,Integrazione condizionata quando l'uscita è limitata
8,use_iob,True,-,Attiva il tetto sull'insulina attiva (IOB)
9,iob_alpha,1.5,adimensionale,Tetto IOB = iob_alpha * basale orario


### Unità di misura delle variabili salvate
| Variabile | Unità | Significato |
|---|---|---|
| `t_h` | h | ora del giorno (la simulazione parte alle 00:00) |
| `CGM` | mg/dL | misura del sensore, con rumore: è l'unica grandezza vista dal controllore |
| `BG` | mg/dL | glicemia plasmatica del modello UVa/Padova (Gp/Vg), usata per le metriche |
| `Insulin_Uh` | U/h | velocità di infusione comandata |
| `IOB` | U | insulina attiva netta stimata dal controllore (eccesso rispetto al basale) |
| `Vincolo` | - | vincolo attivo nel passo: nessuno, `iob`, `plgs`, `sat_alta`, `sat_bassa` |

### Indicatori aggiunti (calcolati in `aid_sim.metrics`, finestra 08:00-24:00)
| Indicatore | Unità | Definizione |
|---|---|---|
| `IAE` | mg/dL·h | integrale dell'errore assoluto rispetto al target: $\int |BG(t) - 120|\,dt$ |
| `IAE_post6h` | mg/dL·h | come IAE, nelle 6 ore dopo il pasto |
| `IAE_range` | mg/dL·h | integrale della distanza dall'intervallo 70-180 (zero dentro il range): penalizza solo le escursioni cliniche |
| `t_rientro` | h | tempo dal pasto al rientro definitivo in 70-180 mg/dL (0 se non esce, NaN se a mezzanotte è ancora fuori) |
| `n_commut` | - | numero di cambi del vincolo attivo (attivazioni e disattivazioni di tetto IOB, PLGS, saturazioni) |
| `n_sosp` | - | numero di sospensioni dell'infusione (passaggi da u > 0 a u = 0) |
| `TV_u` | U/h | variazione totale dell'infusione $\sum_k |u_k - u_{k-1}|$ |
| `std_u` | U/h | deviazione standard dell'infusione |

`n_commut` e `n_sosp` misurano le commutazioni introdotte dai vincoli; `TV_u` misura anche le variazioni continue
del PID dovute al rumore del sensore, quindi è diversa da zero anche senza vincoli.

## 2. Indicatori aggiunti per modulo

In [4]:
MOSTRA = ["TIR", "TIR_post6h", "TAR", "TBR", "TBR_tardivo", "picco", "nadir",
         "IAE", "IAE_post6h", "IAE_range", "t_rientro", "n_commut", "n_sosp", "TV_u", "std_u"]
NOMI = ["PID", "PID+IOB", "PID+PLGS", "PID+IOB+PLGS"]
keys = [RC.key(p, c, n) for p in PAZ for c in CARICHI for n in NOMI]
RC.run_all(keys)
T = pd.DataFrame([{"Paziente": k[0], "CHO [g]": int(k[1]), "Controllore": k[2],
                   **{m: round(RC.cache[k][1][m], 1) for m in MOSTRA}} for k in keys])
display(T)

,Paziente,CHO [g],Controllore,TIR,TIR_post6h,TAR,TBR,TBR_tardivo,picco,nadir,IAE,IAE_post6h,IAE_range,t_rientro,n_commut,n_sosp,TV_u,std_u
0,adult#001,60,PID,93.4,82.5,6.6,0.0,0.0,189.1,70.9,566.0,267.7,6.4,4.7,0,0,9.8,0.8
1,adult#001,60,PID+IOB,92.8,80.8,7.2,0.0,0.0,189.4,76.6,534.3,271.3,6.7,4.8,40,0,73.9,0.8
2,adult#001,60,PID+PLGS,77.2,39.2,22.8,0.0,0.0,207.1,73.9,633.3,357.0,46.1,5.2,10,5,19.9,1.2
3,adult#001,60,PID+IOB+PLGS,76.6,37.5,23.4,0.0,0.0,207.0,80.3,598.2,361.8,47.4,5.3,68,5,118.0,1.1
4,adult#001,90,PID,61.6,25.0,28.1,10.3,18.3,226.1,64.8,758.8,431.2,111.3,NaN,0,0,10.6,0.9
5,adult#001,90,PID+IOB,70.9,22.5,29.1,0.0,0.0,227.1,74.5,718.4,440.7,111.9,5.6,64,0,121.3,0.8
6,adult#001,90,PID+PLGS,61.6,15.8,31.6,6.9,12.2,245.5,69.1,830.4,527.2,197.3,NaN,10,5,20.1,1.3
7,adult#001,90,PID+IOB+PLGS,66.6,13.3,33.4,0.0,0.0,246.8,78.9,799.0,539.9,209.2,6.1,88,4,162.2,1.1
8,adult#003,60,PID,60.3,50.0,18.8,20.9,37.2,238.5,48.6,759.1,397.1,165.0,9.6,0,0,9.8,1.1
9,adult#003,60,PID+IOB,59.4,30.8,25.9,14.7,26.1,238.5,62.0,755.7,430.0,140.0,9.9,56,0,120.0,1.0


## 3. Commutazioni dell'infusione
Infusione comandata con il vincolo attivo evidenziato, pasto da 60 g.

In [5]:
VCOL = {"iob": "tab:purple", "plgs": "tab:green", "sat_alta": "tab:red", "sat_bassa": "tab:gray"}
fig, ax = plt.subplots(len(NOMI), 2, figsize=(14, 10), sharex=True)
for j, p in enumerate(PAZ):
    ubp = A.basal_rate(A.make_patient(p))
    for i, n in enumerate(NOMI):
        d = RC.cache[RC.key(p, 60, n)][0]; d = d[d.t_h >= A.WARMUP_H]; a = ax[i, j]
        a.plot(d.t_h, d.Insulin_Uh, color=COL[n], drawstyle="steps-post", lw=1)
        for v, c in VCOL.items():
            on = (d.Vincolo == v).values
            if on.any():
                a.fill_between(d.t_h, 0, 1, where=on, step="post", color=c, alpha=0.25,
                               transform=a.get_xaxis_transform(), label=f"vincolo: {v}")
        a.axhline(ubp * 60, color="gray", ls=":", lw=1); a.axvline(A.MEAL_CLOCK_H, color="k", ls=":")
        m = RC.cache[RC.key(p, 60, n)][1]
        a.set_title(f"{p} - {n}: {m['n_commut']} commutazioni, {m['n_sosp']} sospensioni", fontsize=9)
        a.set_ylabel("U/h"); a.grid(ls=":", alpha=0.5)
        if a.get_legend_handles_labels()[0]: a.legend(fontsize=7, loc="upper right")
    ax[-1, j].set_xlabel("Ora del giorno [h]")
plt.tight_layout(); plt.show()

## 4. Sensibilità ai parametri del tetto IOB e del PLGS
Un parametro alla volta sul controllore `PID+IOB+PLGS`, gli altri al valore nominale.

| Parametro | Unità | Valori provati | Nominale |
|---|---|---|---|
| `iob_alpha` (tetto IOB = α · basale orario) | - | 1, 1.5, 2, 3 | 1.5 |
| `tau_s` (costante di tempo dello stimatore IOB) | min | 30, 50, 70 | 50 |
| `t_pred` (orizzonte di predizione del PLGS) | min | 10, 20, 30 | 20 |
| `plgs_pred` (soglia sulla glicemia proiettata) | mg/dL | 80, 90, 100 | 90 |

In [6]:
sk = [RC.key(p, c, "PID+IOB+PLGS", **{par: v}) for par, vals in RC.SENS.items() for v in vals
      for p in PAZ for c in CARICHI]
RC.run_all(sk)
S = pd.DataFrame([{"Parametro": k[6][0][0], "Valore": k[6][0][1], "Paziente": k[0], "CHO [g]": int(k[1]),
                   **{m: round(RC.cache[k][1][m], 1) for m in ["TIR", "TAR", "TBR", "TBR_tardivo", "picco",
                                                              "IAE_range", "t_rientro", "n_commut", "TV_u"]}}
                  for k in sk])
for par in RC.SENS:
    print(par); display(S[S.Parametro == par].drop(columns="Parametro").reset_index(drop=True))

iob_alpha


,Valore,Paziente,CHO [g],TIR,TAR,TBR,TBR_tardivo,picco,IAE_range,t_rientro,n_commut,TV_u
0,1.0,adult#001,60,80.6,19.4,0.0,0.0,200.8,27.4,5.2,145,216.4
1,1.0,adult#001,90,65.9,34.1,0.0,0.0,241.6,188.9,6.2,156,250.8
2,1.0,adult#003,60,68.8,31.2,0.0,0.0,220.5,97.1,4.8,122,233.9
3,1.0,adult#003,90,63.4,36.6,0.0,0.0,257.4,296.7,5.6,139,272.0
4,1.5,adult#001,60,76.6,23.4,0.0,0.0,207.0,47.4,5.3,68,118.0
5,1.5,adult#001,90,66.6,33.4,0.0,0.0,246.8,209.2,6.1,88,162.2
6,1.5,adult#003,60,78.1,21.9,0.0,0.0,214.7,55.0,2.6,69,136.1
7,1.5,adult#003,90,62.8,37.2,0.0,0.0,244.8,178.1,5.1,86,182.3
8,2.0,adult#001,60,76.9,23.1,0.0,0.0,207.1,46.2,5.2,24,46.3
9,2.0,adult#001,90,67.8,32.2,0.0,0.0,245.6,199.8,5.9,38,76.8


tau_s


,Valore,Paziente,CHO [g],TIR,TAR,TBR,TBR_tardivo,picco,IAE_range,t_rientro,n_commut,TV_u
0,30.0,adult#001,60,77.2,22.8,0.0,0.0,207.1,46.1,5.2,12,23.9
1,30.0,adult#001,90,68.4,31.6,0.0,0.0,245.5,197.3,5.9,18,37.3
2,30.0,adult#003,60,63.4,27.2,9.4,16.7,214.2,62.1,NaN,3,12.7
3,30.0,adult#003,90,60.6,30.0,9.4,16.7,242.9,129.6,NaN,5,18.4
4,50.0,adult#001,60,76.6,23.4,0.0,0.0,207.0,47.4,5.3,68,118.0
5,50.0,adult#001,90,66.6,33.4,0.0,0.0,246.8,209.2,6.1,88,162.2
6,50.0,adult#003,60,78.1,21.9,0.0,0.0,214.7,55.0,2.6,69,136.1
7,50.0,adult#003,90,62.8,37.2,0.0,0.0,244.8,178.1,5.1,86,182.3
8,70.0,adult#001,60,77.5,22.5,0.0,0.0,203.8,37.8,5.2,126,189.7
9,70.0,adult#001,90,65.9,34.1,0.0,0.0,243.9,201.1,6.2,142,230.3


t_pred


,Valore,Paziente,CHO [g],TIR,TAR,TBR,TBR_tardivo,picco,IAE_range,t_rientro,n_commut,TV_u
0,10.0,adult#001,60,78.1,21.9,0.0,0.0,204.8,38.8,5.2,62,106.8
1,10.0,adult#001,90,66.9,33.1,0.0,0.0,244.3,196.2,6.1,86,158.1
2,10.0,adult#003,60,75.3,24.7,0.0,0.0,218.5,66.9,2.7,71,139.7
3,10.0,adult#003,90,60.3,39.7,0.0,0.0,248.5,193.7,5.1,89,185.8
4,20.0,adult#001,60,76.6,23.4,0.0,0.0,207.0,47.4,5.3,68,118.0
5,20.0,adult#001,90,66.6,33.4,0.0,0.0,246.8,209.2,6.1,88,162.2
6,20.0,adult#003,60,78.1,21.9,0.0,0.0,214.7,55.0,2.6,69,136.1
7,20.0,adult#003,90,62.8,37.2,0.0,0.0,244.8,178.1,5.1,86,182.3
8,30.0,adult#001,60,75.0,25.0,0.0,0.0,208.8,55.8,5.4,70,124.2
9,30.0,adult#001,90,65.9,34.1,0.0,0.0,248.8,221.3,6.2,94,174.1


plgs_pred


,Valore,Paziente,CHO [g],TIR,TAR,TBR,TBR_tardivo,picco,IAE_range,t_rientro,n_commut,TV_u
0,80.0,adult#001,60,90.3,9.7,0.0,0.0,195.4,14.7,4.9,55,92.3
1,80.0,adult#001,90,69.1,30.9,0.0,0.0,233.8,141.4,5.9,81,144.0
2,80.0,adult#003,60,84.1,15.9,0.0,0.0,220.0,65.5,2.8,71,139.3
3,80.0,adult#003,90,70.0,30.0,0.0,0.0,252.1,201.0,5.1,89,187.9
4,90.0,adult#001,60,76.6,23.4,0.0,0.0,207.0,47.4,5.3,68,118.0
5,90.0,adult#001,90,66.6,33.4,0.0,0.0,246.8,209.2,6.1,88,162.2
6,90.0,adult#003,60,78.1,21.9,0.0,0.0,214.7,55.0,2.6,69,136.1
7,90.0,adult#003,90,62.8,37.2,0.0,0.0,244.8,178.1,5.1,86,182.3
8,100.0,adult#001,60,71.6,28.4,0.0,0.0,215.7,91.3,5.5,80,140.6
9,100.0,adult#001,90,64.7,35.3,0.0,0.0,256.7,268.7,6.4,100,190.3


In [7]:
MET = [("TIR", "TIR [%]"), ("TBR", "TBR [%]"), ("picco", "Picco [mg/dL]"), ("n_commut", "Commutazioni [-]")]
fig, ax = plt.subplots(len(MET), len(RC.SENS), figsize=(16, 11), sharex="col")
for j, (par, vals) in enumerate(RC.SENS.items()):
    for p in PAZ:
        for c, ls in zip(CARICHI, ["-", "--"]):
            for i, (m, lab) in enumerate(MET):
                y = [RC.cache[RC.key(p, c, "PID+IOB+PLGS", **{par: v})][1][m] for v in vals]
                line, = ax[i, j].plot(vals, y, marker="o", ls=ls, label=f"{p}, {c} g")
                ax[i, j].axhline(RC.cache[RC.key(p, c, "PID")][1][m], color=line.get_color(), ls=ls, lw=0.8, alpha=0.4)
    for i, (m, lab) in enumerate(MET):
        ax[i, j].axvline(RC.NOMINALE[par], color="k", ls=":", lw=1); ax[i, j].grid(ls=":", alpha=0.5)
        if j == 0: ax[i, j].set_ylabel(lab)
    ax[-1, j].set_xlabel(par)
ax[0, 0].legend(fontsize=7)
fig.suptitle("Linea verticale: valore nominale; linee orizzontali sottili: PID senza vincoli")
plt.tight_layout(); plt.show()

## 5. Lettura dei risultati (calcolata dalle tabelle)

In [8]:
for p in PAZ:
    for c in CARICHI:
        r = {n: T[(T.Paziente == p) & (T["CHO [g]"] == c) & (T.Controllore == n)].iloc[0] for n in NOMI}
        print(f"{p} {c} g: rispetto al PID")
        for n in NOMI[1:]:
            print(f"   {n:13s} dTIR {r[n].TIR - r['PID'].TIR:+6.1f}  dTBR {r[n].TBR - r['PID'].TBR:+6.1f}  "
                  f"dpicco {r[n].picco - r['PID'].picco:+6.1f}  dIAE_range {r[n].IAE_range - r['PID'].IAE_range:+7.1f}  "
                  f"commutazioni {int(r[n].n_commut):3d}  sospensioni {int(r[n].n_sosp):3d}")
print()
for par, vals in RC.SENS.items():
    d = S[S.Parametro == par]
    rng = d.groupby(["Paziente", "CHO [g]"])[["TIR", "TBR", "picco"]].agg(lambda x: x.max() - x.min()).max()
    print(f"{par:10s} escursione massima al variare del parametro: TIR {rng.TIR:5.1f} punti, "
          f"TBR {rng.TBR:5.1f} punti, picco {rng.picco:5.1f} mg/dL")

adult#001 60 g: rispetto al PID
   PID+IOB       dTIR   -0.6  dTBR   +0.0  dpicco   +0.3  dIAE_range    +0.3  commutazioni  40  sospensioni   0
   PID+PLGS      dTIR  -16.2  dTBR   +0.0  dpicco  +18.0  dIAE_range   +39.7  commutazioni  10  sospensioni   5
   PID+IOB+PLGS  dTIR  -16.8  dTBR   +0.0  dpicco  +17.9  dIAE_range   +41.0  commutazioni  68  sospensioni   5
adult#001 90 g: rispetto al PID
   PID+IOB       dTIR   +9.3  dTBR  -10.3  dpicco   +1.0  dIAE_range    +0.6  commutazioni  64  sospensioni   0
   PID+PLGS      dTIR   +0.0  dTBR   -3.4  dpicco  +19.4  dIAE_range   +86.0  commutazioni  10  sospensioni   5
   PID+IOB+PLGS  dTIR   +5.0  dTBR  -10.3  dpicco  +20.7  dIAE_range   +97.9  commutazioni  88  sospensioni   4
adult#003 60 g: rispetto al PID
   PID+IOB       dTIR   -0.9  dTBR   -6.2  dpicco   +0.0  dIAE_range   -25.0  commutazioni  56  sospensioni   0
   PID+PLGS      dTIR   +3.1  dTBR  -11.5  dpicco  -24.3  dIAE_range  -102.9  commutazioni   3  sospensioni   1
   PID+I

**Note per l'interpretazione.**
- Il confronto `PID+IOB` / `PID+PLGS` / `PID+IOB+PLGS` separa il contributo dei due moduli a parità di PID.
- Il tetto IOB è un interruttore (sopra il tetto l'infusione torna di colpo al basale) e il PLGS sospende e riprende
  l'infusione: entrambi introducono commutazioni, contate in `n_commut` e `n_sosp`.
- La sensibilità è fatta un parametro alla volta: non esplora le interazioni tra parametri e non è un'ottimizzazione.